In [12]:
import math
import re
import getpass

In [13]:
common_passwords = {
    "123456",
    "123456789",
    "password",
    "password123",
    "qwerty",
    "qwerty123",
    "admin",
    "admin123",
    "letmein",
    "welcome",
    "abc123",
    "iloveyou",
    "football",
    "monkey"
}

print(f"Loaded {len(common_passwords)} common passwords.")

Loaded 14 common passwords.


In [14]:
def analyze_characters(password):
    analysis = {
        "length": len(password),
        "has_lower": bool(re.search(r"[a-z]", password)),
        "has_upper": bool(re.search(r"[A-Z]", password)),
        "has_digit": bool(re.search(r"\d", password)),
        "has_symbol": bool(re.search(r"[^A-Za-z0-9]", password))
    }

    return analysis

In [15]:
test_password = "Hello123!"

analysis = analyze_characters(test_password)

print("Password Analysis")
print("-----------------")
print("Length:", analysis["length"])
print("Lowercase:", analysis["has_lower"])
print("Uppercase:", analysis["has_upper"])
print("Digit:", analysis["has_digit"])
print("Symbol:", analysis["has_symbol"])

Password Analysis
-----------------
Length: 9
Lowercase: True
Uppercase: True
Digit: True
Symbol: True


In [16]:
def calculate_score(password, analysis):

    score = 0

    # Length score
    if analysis["length"] >= 16:
        score += 3
    elif analysis["length"] >= 12:
        score += 2
    elif analysis["length"] >= 8:
        score += 1

    # Character diversity
    if analysis["has_lower"]:
        score += 1

    if analysis["has_upper"]:
        score += 1

    if analysis["has_digit"]:
        score += 1

    if analysis["has_symbol"]:
        score += 1

    # Common password check
    is_common = password.lower() in common_passwords

    if is_common:
        score = min(score, 2)

    return score, is_common

In [17]:
def classify_strength(score):

    if score <= 3:
        return "Weak"

    elif score <= 5:
        return "Medium"

    else:
        return "Strong"

In [18]:
def generate_recommendations(analysis, is_common):

    recommendations = []

    if analysis["length"] < 12:
        recommendations.append(
            "Increase the password length to at least 12 characters."
        )

    if not analysis["has_lower"]:
        recommendations.append(
            "Add lowercase letters."
        )

    if not analysis["has_upper"]:
        recommendations.append(
            "Add uppercase letters."
        )

    if not analysis["has_digit"]:
        recommendations.append(
            "Add numbers."
        )

    if not analysis["has_symbol"]:
        recommendations.append(
            "Add special characters."
        )

    if is_common:
        recommendations.append(
            "Avoid common or easily guessable passwords."
        )

    return recommendations

In [19]:
def calculate_entropy(password, analysis):

    pool_size = 0

    if analysis["has_lower"]:
        pool_size += 26

    if analysis["has_upper"]:
        pool_size += 26

    if analysis["has_digit"]:
        pool_size += 10

    if analysis["has_symbol"]:
        pool_size += 32

    if pool_size == 0:
        return 0

    entropy = len(password) * math.log2(pool_size)

    return entropy

In [20]:
def estimate_crack_time(entropy):

    guesses_per_second = 1_000_000_000

    combinations = 2 ** entropy

    # Average brute-force search would find the password
    # after approximately half the possible combinations.
    seconds = (combinations / 2) / guesses_per_second

    if seconds < 1:
        return "Less than 1 second"

    elif seconds < 60:
        return f"{seconds:.2f} seconds"

    elif seconds < 3600:
        return f"{seconds / 60:.2f} minutes"

    elif seconds < 86400:
        return f"{seconds / 3600:.2f} hours"

    elif seconds < 31536000:
        return f"{seconds / 86400:.2f} days"

    else:
        years = seconds / 31536000

        if years > 1_000_000:
            return "More than 1 million years"

        return f"{years:,.2f} years"

In [21]:
def check_password(password):

    analysis = analyze_characters(password)

    score, is_common = calculate_score(
        password,
        analysis
    )

    strength = classify_strength(score)

    recommendations = generate_recommendations(
        analysis,
        is_common
    )

    entropy = calculate_entropy(
        password,
        analysis
    )

    crack_time = estimate_crack_time(entropy)

    return {
        "analysis": analysis,
        "score": score,
        "strength": strength,
        "is_common": is_common,
        "recommendations": recommendations,
        "entropy": entropy,
        "crack_time": crack_time
    }

In [22]:
password = getpass.getpass("Enter a password to analyze: ")

if not password:
    print("\nPassword cannot be empty.")

else:
    result = check_password(password)

    print("\n" + "=" * 45)
    print("        PASSWORD SECURITY REPORT")
    print("=" * 45)

    print(f"\nStrength: {result['strength']}")
    print(f"Score: {result['score']}/7")

    print("\nPassword Analysis:")

    analysis = result["analysis"]

    print(
        "✓" if analysis["length"] >= 12 else "✗",
        f"Length: {analysis['length']} characters"
    )

    print(
        "✓" if analysis["has_lower"] else "✗",
        "Lowercase letters"
    )

    print(
        "✓" if analysis["has_upper"] else "✗",
        "Uppercase letters"
    )

    print(
        "✓" if analysis["has_digit"] else "✗",
        "Numbers"
    )

    print(
        "✓" if analysis["has_symbol"] else "✗",
        "Special characters"
    )

    print(
        "✗" if result["is_common"] else "✓",
        "Common-password check"
    )

    print(f"\nEstimated Entropy: {result['entropy']:.2f} bits")
    print(
        "Estimated Theoretical Crack Time:",
        result["crack_time"]
    )

    print("\nRecommendations:")

    if result["recommendations"]:
        for recommendation in result["recommendations"]:
            print("-", recommendation)
    else:
        print("- No basic weaknesses detected by this checker.")

    print("\n" + "=" * 45)

Enter a password to analyze: ··········

        PASSWORD SECURITY REPORT

Strength: Weak
Score: 1/7

Password Analysis:
✗ Length: 6 characters
✗ Lowercase letters
✗ Uppercase letters
✓ Numbers
✗ Special characters
✗ Common-password check

Estimated Entropy: 19.93 bits
Estimated Theoretical Crack Time: Less than 1 second

Recommendations:
- Increase the password length to at least 12 characters.
- Add lowercase letters.
- Add uppercase letters.
- Add special characters.
- Avoid common or easily guessable passwords.

